### Needs `01-ReferenceDFs.ipynb` to be executed first

Reads from:
- `PipelineB_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np
import random

## Import Data

In [2]:
# As this was the first evaluation script of this kind, I kept the non-ynom analysis for reasoning why we moved to ynorm as the standard
answers  = pd.read_json("PipelineB.json")
answers_ynorm = pd.read_json("PipelineB_ynorm.json")
CATEGORIES = ["value", "unit"]
VARIANTS   = ["t1", "t2", "m1", "m2", "i1", "i2", "g"]

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    answers[col] = answers[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    answers_ynorm[col] = answers_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [4]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()
    
    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()
    
    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set


def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
                
    return result

def eval(source, exact) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v:<2}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## Straight Forward

In [5]:
just_eval, expended = eval(answers, False)

print_eval(just_eval)

### value ###
t1: True = 2078 || False = 130 || Quota = 94.11%
t2: True = 2087 || False = 121 || Quota = 94.52%
m1: True = 1607 || False = 601 || Quota = 72.78%
m2: True = 1923 || False = 285 || Quota = 87.09%
i1: True = 1874 || False = 334 || Quota = 84.87%
i2: True = 1858 || False = 350 || Quota = 84.15%
g : True = 2065 || False = 143 || Quota = 93.52%

### unit ###
t1: True = 1952 || False = 256 || Quota = 88.41%
t2: True = 1935 || False = 273 || Quota = 87.64%
m1: True = 1764 || False = 444 || Quota = 79.89%
m2: True = 1787 || False = 421 || Quota = 80.93%
i1: True = 1761 || False = 447 || Quota = 79.76%
i2: True = 1744 || False = 464 || Quota = 78.99%
g : True = 1930 || False = 278 || Quota = 87.41%



### Exact

In [6]:
just_eval_exact, _ = eval(answers, True)

print_eval(just_eval_exact)

### value ###
t1: True = 2057 || False = 151 || Quota = 93.16%
t2: True = 2056 || False = 152 || Quota = 93.12%
m1: True = 1607 || False = 601 || Quota = 72.78%
m2: True = 1909 || False = 299 || Quota = 86.46%
i1: True = 1851 || False = 357 || Quota = 83.83%
i2: True = 1820 || False = 388 || Quota = 82.43%
g : True = 2051 || False = 157 || Quota = 92.89%

### unit ###
t1: True = 1943 || False = 265 || Quota = 88.0%
t2: True = 1926 || False = 282 || Quota = 87.23%
m1: True = 1755 || False = 453 || Quota = 79.48%
m2: True = 1785 || False = 423 || Quota = 80.84%
i1: True = 1755 || False = 453 || Quota = 79.48%
i2: True = 1735 || False = 473 || Quota = 78.58%
g : True = 1927 || False = 281 || Quota = 87.27%



## YNROM

In [7]:
ynorm, expended_ynorm = eval(answers_ynorm, False)

print_eval(ynorm)

### value ###
t1: True = 2099 || False = 109 || Quota = 95.06%
t2: True = 2115 || False = 93 || Quota = 95.79%
m1: True = 1591 || False = 617 || Quota = 72.06%
m2: True = 1935 || False = 273 || Quota = 87.64%
i1: True = 1899 || False = 309 || Quota = 86.01%
i2: True = 1885 || False = 323 || Quota = 85.37%
g : True = 2139 || False = 69 || Quota = 96.88%

### unit ###
t1: True = 1973 || False = 235 || Quota = 89.36%
t2: True = 1964 || False = 244 || Quota = 88.95%
m1: True = 1763 || False = 445 || Quota = 79.85%
m2: True = 1794 || False = 414 || Quota = 81.25%
i1: True = 1785 || False = 423 || Quota = 80.84%
i2: True = 1768 || False = 440 || Quota = 80.07%
g : True = 1998 || False = 210 || Quota = 90.49%



### Exact

In [8]:
ynorm_exact, forGEPA = eval(answers_ynorm, True)

print_eval(ynorm_exact)

### value ###
t1: True = 2078 || False = 130 || Quota = 94.11%
t2: True = 2084 || False = 124 || Quota = 94.38%
m1: True = 1591 || False = 617 || Quota = 72.06%
m2: True = 1921 || False = 287 || Quota = 87.0%
i1: True = 1860 || False = 348 || Quota = 84.24%
i2: True = 1826 || False = 382 || Quota = 82.7%
g : True = 2124 || False = 84 || Quota = 96.2%

### unit ###
t1: True = 1964 || False = 244 || Quota = 88.95%
t2: True = 1955 || False = 253 || Quota = 88.54%
m1: True = 1754 || False = 454 || Quota = 79.44%
m2: True = 1792 || False = 416 || Quota = 81.16%
i1: True = 1779 || False = 429 || Quota = 80.57%
i2: True = 1759 || False = 449 || Quota = 79.66%
g : True = 1995 || False = 213 || Quota = 90.35%



# Detailed Comparison

In [ ]:
def eval_comparison(eval_raw, eval_ynorm, filepath="eval_comparison.json"):
    """Speichert beide Evaluationen strukturiert zum Vergleich"""
    results = []
    
    for cat in CATEGORIES:
        for v in VARIANTS:
            col_raw = eval_raw[f"{v}_{cat}_hit"]
            col_ynorm = eval_ynorm[f"{v}_{cat}_hit"]
            
            results.append({
                "variant": v,
                "category": cat,
                "raw_true": col_raw.value_counts()[True],
                "ynorm_true": col_ynorm.value_counts()[True],
                "raw_false": col_raw.value_counts()[False],
                "ynorm_false": col_ynorm.value_counts()[False],
                "abs_improvmnt": col_ynorm.value_counts()[True] - col_raw.value_counts()[True],
                "raw_quota": round(col_raw.mean() * 100, 2),
                "ynorm_quota": round(col_ynorm.mean() * 100, 2),
                "delta_quota": round((col_ynorm.mean() - col_raw.mean()) * 100, 2)
            })
    
    df_results = pd.DataFrame(results)
    #df_results.to_json(filepath, orient="records", indent=2)
    #df_results.to_csv(filepath.replace(".json", ".csv"), index=False)
    
    return df_results

# Speichern und anzeigen
comparison = eval_comparison(just_eval, ynorm)
print(comparison)


   variant category  raw_true  ynorm_true  raw_false  ynorm_false  \
0       t1    value      2078        2099        130          109   
1       t2    value      2087        2115        121           93   
2       m1    value      1607        1591        601          617   
3       m2    value      1923        1935        285          273   
4       i1    value      1874        1899        334          309   
5       i2    value      1858        1885        350          323   
6        g    value      2065        2139        143           69   
7       t1     unit      1952        1973        256          235   
8       t2     unit      1935        1964        273          244   
9       m1     unit      1764        1763        444          445   
10      m2     unit      1787        1794        421          414   
11      i1     unit      1761        1785        447          423   
12      i2     unit      1744        1768        464          440   
13       g     unit      1930     

In [ ]:
comparison_exact = eval_comparison(just_eval_exact, ynorm_exact)
print(comparison_exact)

   variant category  raw_true  ynorm_true  raw_false  ynorm_false  \
0       t1    value      2057        2078        151          130   
1       t2    value      2056        2084        152          124   
2       m1    value      1607        1591        601          617   
3       m2    value      1909        1921        299          287   
4       i1    value      1851        1860        357          348   
5       i2    value      1820        1826        388          382   
6        g    value      2051        2124        157           84   
7       t1     unit      1943        1964        265          244   
8       t2     unit      1926        1955        282          253   
9       m1     unit      1755        1754        453          454   
10      m2     unit      1785        1792        423          416   
11      i1     unit      1755        1779        453          429   
12      i2     unit      1735        1759        473          449   
13       g     unit      1927     

In [11]:
# No merge necessary as the two DFs have the same index
comparison_exact["raw_true_any"] = comparison["raw_true"]
comparison_exact["fewer_hits_raw"] = comparison_exact["raw_true"] - comparison_exact["raw_true_any"]
comparison_exact["ynorm_true_any"] = comparison["ynorm_true"]
comparison_exact["fewer_hits_ynorm"] = comparison_exact["ynorm_true"] - comparison_exact["ynorm_true_any"]
comparison_exact[["variant","category","raw_true","ynorm_true","raw_true_any","fewer_hits_raw","ynorm_true_any","fewer_hits_ynorm"]]

,variant,category,raw_true,ynorm_true,raw_true_any,fewer_hits_raw,ynorm_true_any,fewer_hits_ynorm
0,t1,value,2057,2078,2078,-21,2099,-21
1,t2,value,2056,2084,2087,-31,2115,-31
2,m1,value,1607,1591,1607,0,1591,0
3,m2,value,1909,1921,1923,-14,1935,-14
4,i1,value,1851,1860,1874,-23,1899,-39
5,i2,value,1820,1826,1858,-38,1885,-59
6,g,value,2051,2124,2065,-14,2139,-15
7,t1,unit,1943,1964,1952,-9,1973,-9
8,t2,unit,1926,1955,1935,-9,1964,-9
9,m1,unit,1755,1754,1764,-9,1763,-9


* You can see that the hits have dropped in general. Interestingly not all equal, not even for the same model.
* The drops are more evenly distributed for units than for values.


# Stuff for initial GEPA run with the instruct model

In [12]:
just_i = answers_ynorm[
    ['report_name',
    'status',
    'scopes_present',
    'years_present',
    'year',
    'scope',
    'page',
    'value',
    'unit',
    'unit_normalized',
    'label',
    'value_i1',
    'value_i2',
    'unit_i1',
    'unit_i2',
    'label_i1',
    'label_i2']
].copy()

In [13]:
VARIANTS = ["i1", "i2"] # Just for this part, to only get the intr. cases
# Exact comparison!

eval_i, eval_i_expended = eval(just_i, True)

print_eval(eval_i)

VARIANTS   = ["t1", "t2", "m1", "m2", "i1", "i2", "g"] # switching back to original values

### value ###
i1: True = 1860 || False = 348 || Quota = 84.24%
i2: True = 1826 || False = 382 || Quota = 82.7%

### unit ###
i1: True = 1779 || False = 429 || Quota = 80.57%
i2: True = 1759 || False = 449 || Quota = 79.66%



### Choosing 60% Dataset for GEPA with Tinking-Model

In [35]:
_, forGEPA = eval(answers_ynorm, True) #Fresh Calculation

forGEPA = forGEPA[["report_name","t1_value_hit","t2_value_hit"]]
print(forGEPA)

                        report_name  t1_value_hit  t2_value_hit
0     acuity brands inc_2022_report          True          True
1     acuity brands inc_2022_report          True          True
2     acuity brands inc_2022_report          True          True
3     acuity brands inc_2022_report          True          True
4     acuity brands inc_2022_report          True          True
...                             ...           ...           ...
2203            webuild_2019_report          True          True
2204            webuild_2019_report          True          True
2205            webuild_2019_report          True          True
2206            webuild_2019_report          True          True
2207            webuild_2019_report          True          True

[2208 rows x 3 columns]


In [36]:
t1 = forGEPA["t1_value_hit"]
t2 = forGEPA["t2_value_hit"]

not_match = (t1 != t2).sum()
t1_true_t2_false = (t1 & ~t2).sum()
t1_false_t2_true = (~t1 & t2).sum()

print(f"t1 vs t2 not matching: {not_match}")
print(f"t1=True, t2=False: {t1_true_t2_false}")
print(f"t1=False, t2=True: {t1_false_t2_true}")

t1 vs t2 not matching: 68
t1=True, t2=False: 31
t1=False, t2=True: 37


==> Seems pretty much 50/50 so no run was really better or worse. Therefore, dropping t2.

In [37]:
forGEPA = forGEPA[["report_name","t1_value_hit"]]

In [38]:
hits = forGEPA.groupby("report_name")["t1_value_hit"].all()
GEPA_TRUE  = hits[hits == True].index.tolist()
GEPA_FALSE = hits[hits == False].index.tolist()

print(len(GEPA_TRUE))
print(len(GEPA_FALSE))

25
29


In [39]:
random.seed(42)

train_true  = random.sample(GEPA_TRUE,  k=round(len(GEPA_TRUE)  * 0.6))
train_false = random.sample(GEPA_FALSE, k=round(len(GEPA_FALSE) * 0.6))

eval_true  = [r for r in GEPA_TRUE  if r not in train_true]
eval_false = [r for r in GEPA_FALSE if r not in train_false]

train = train_true + train_false
eval_ = eval_true  + eval_false

print(f"Train: {len(train)} ({len(train_true)} TRUE, {len(train_false)} FALSE)")
print(f"Eval:  {len(eval_)} ({len(eval_true)} TRUE, {len(eval_false)} FALSE)")

Train: 32 (15 TRUE, 17 FALSE)
Eval:  22 (10 TRUE, 12 FALSE)


In [40]:
sorted(train)

['Allianz_2022_report',
 'Fresenius SE_2019_report',
 'ViacomCBS_ESG Report_2020-2021_vFINAL',
 'acuity brands inc_2022_report',
 'addtech_2022_report',
 'aixtron_2020_report',
 'autoneum holding_2019_report',
 'bekaert (d) sa_2022_report',
 'cabot corp_2018_report',
 'cardinal energy ltd_2021_report',
 'dampskibsselskabet norden_2019_report',
 'evolution mining ltd_2020_report',
 'granite construction inc_2020_report',
 'hammerson reit_2021_report',
 'hang lung properties_2018_report',
 'healius ltd_2022_report',
 'huhtamaki_2018_report',
 'huhtamaki_2019_report',
 'inchcape plc_2022_report',
 'innospec inc_2020_report',
 'jetblue airways corp_2019_report',
 'kilroy realty_2017_report',
 'kilroy realty_2018_report',
 'lundin gold inc_2021_report',
 'nippn corp_2022_report',
 'nordea_2017_report',
 'polaris_2021_report',
 'salzgitter ag_2018_report',
 'stantec inc_2019_report',
 'sumitomo corporation_2021_report',
 'varta ag_2021_report',
 'webuild_2019_report']

# GEPA Evaluation
vs. Thinking 1
##  (EXACT)

In [41]:
_, gepa_eval_exact = eval(answers_ynorm, True) #Fresh Calculation

In [42]:
gepa_eval_exact = gepa_eval_exact[['report_name', 'year', 'scope', 'value', 'unit', 'unit_normalized', 'label', 'value_t1', 'value_g', 'unit_t1', 'unit_g', 'label_t1', 'label_g', 't1_value_hit','g_value_hit','t1_unit_hit','g_unit_hit']]

In [43]:
gepa_improvement_exact = (gepa_eval_exact["t1_value_hit"] == False) & (gepa_eval_exact["g_value_hit"] == True)
gepa_eval_exact.loc[gepa_improvement_exact, ["report_name", "year", "scope", "value", "value_t1", "value_g"]]

,report_name,year,scope,value,value_t1,value_g
174,Allianz_2022_report,2020,1,28714.0,"[28714.0, 29.0]",[28714.0]
176,Allianz_2022_report,2021,1,28699.0,"[28699.0, 29.0]",[28699.0]
178,Allianz_2022_report,2022,1,30953.0,"[30953.0, 31.0]",[30953.0]
197,Allianz_2022_report,2020,2mb,100722.0,"[100722.0, 101.0]",[100722.0]
199,Allianz_2022_report,2021,2mb,54689.0,"[54689.0, 55.0]",[54689.0]
...,...,...,...,...,...,...
1969,sumitomo corporation_2021_report,2020,1,1526724.0,"[2179335.0, 1522514.0]",[1526724.0]
2019,toshiba tec corp_2022_report,2020,2lb,48.8,NaN,[48.8]
2020,toshiba tec corp_2022_report,2021,2lb,45.5,NaN,[45.5]
2029,toshiba tec corp_2022_report,2020,2mb,NaN,[48.8],NaN


## (ANY)

In [47]:
_, gepa_eval = eval(answers_ynorm, False) #Frash Calculation

In [48]:
gepa_eval = gepa_eval[['report_name', 'year', 'scope', 'value', 'unit', 'unit_normalized', 'label', 'value_t1', 'value_g', 'unit_t1', 'unit_g', 'label_t1', 'label_g', 't1_value_hit','g_value_hit','t1_unit_hit','g_unit_hit']]

In [49]:
gepa_improvement = (gepa_eval["t1_value_hit"] == False) & (gepa_eval["g_value_hit"] == True)

gepa_eval.loc[gepa_improvement, ["report_name", "year", "scope", "value", "value_t1", "value_g"]]


,report_name,year,scope,value,value_t1,value_g
234,americold realty inc_2022_report,2021,2lb,NaN,[556660.0],NaN
235,americold realty inc_2022_report,2022,2lb,NaN,[554491.0],NaN
342,bekaert (d) sa_2022_report,2019,1,NaN,"[286932.0, 7867.0]",NaN
343,bekaert (d) sa_2022_report,2020,1,NaN,"[263785.0, 7445.0]",NaN
344,bekaert (d) sa_2022_report,2021,1,NaN,"[277023.0, 7887.0]",NaN
...,...,...,...,...,...,...
2030,toshiba tec corp_2022_report,2021,2mb,NaN,[45.5],NaN
2060,uniper_2019_report,2018,2lb,1101032.0,[1.1],"[1.1, 1101032.0]"
2062,uniper_2019_report,2019,2lb,1129277.0,[1.12],"[1.12, 1129277.0]"
2071,uniper_2019_report,2018,2mb,1676689.0,[1.67],"[1.67, 1676689.0]"


#### See, which reports profit from "any" matching, so where multiple values got extracted and therefore "exact" did not match.

In [50]:
better_with_any = (gepa_eval["g_value_hit"] == True) & (gepa_eval_exact["g_value_hit"] == False)
gepa_eval.loc[better_with_any, ["report_name", "year", "scope", "value", "value_g"]]


,report_name,year,scope,value,value_g
776,fujifilm_2022_report,2021,3,3473.00,"[3473.0, 2360.0, 1113.0]"
1189,inchcape plc_2022_report,2021,1,9752.00,"[2486.0, 9752.0]"
1190,inchcape plc_2022_report,2022,1,17002.00,"[3617.0, 17002.0]"
1199,inchcape plc_2022_report,2021,2lb,27277.00,"[3689.0, 27277.0]"
1200,inchcape plc_2022_report,2022,2lb,22223.00,"[2886.0, 22223.0]"
1609,metro ag_2022_report,2020,1,270761.00,"[270761.0, 170163.0]"
1610,metro ag_2022_report,2021,1,264654.80,"[264654.8, 167028.2]"
1611,metro ag_2022_report,2022,1,266631.80,"[266631.8, 167592.4]"
2060,uniper_2019_report,2018,2lb,1101032.00,"[1.1, 1101032.0]"
2061,uniper_2019_report,2019,2lb,1.12,"[1.12, 1129277.0]"
